In [6]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd

In [9]:
from pathlib import Path

data_dir = next(
    (
        path
        for path in (Path("data"), Path("../data"), Path("."))
        if (path / "tmdb_5000_movies.csv").is_file()
        and (path / "tmdb_5000_credits.csv").is_file()
    ),
    None,
)

if data_dir is None:
    raise FileNotFoundError("Could not find the TMDB CSV files. Set data_dir to their folder.")

movies = pd.read_csv(data_dir / "tmdb_5000_movies.csv")
credits = pd.read_csv(data_dir / "tmdb_5000_credits.csv")


In [10]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [11]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [12]:
movies=movies.merge(credits , on='title')

In [13]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [14]:
movies.shape

(4809, 23)

In [15]:
# what columns are useful

#genre
#id
#keyword
#overview
#title
#cast
#crew

movies=movies[['movie_id','title','genres','overview','keywords','cast','crew']]

In [16]:
movies.head(1)

,movie_id,title,genres,overview,keywords,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [17]:

# we will create a new dataset having three columns as - 'movie_id','tags','title'
# in title we will merge the genres,overview,cast and crew all of these 

In [18]:
movies.isnull().sum()

movie_id    0
title       0
genres      0
overview    3
keywords    0
cast        0
crew        0
dtype: int64

In [19]:
movies.dropna(inplace=True)

In [20]:
movies.isnull().sum()

movie_id    0
title       0
genres      0
overview    0
keywords    0
cast        0
crew        0
dtype: int64

In [21]:
movies.duplicated().sum()

np.int64(0)

In [22]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [23]:
# converting the string of the dictionary in the list
import ast

def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L    



In [24]:
convert('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [25]:
movies['genres']=movies['genres'].apply(convert)

In [26]:
movies['keywords']=movies['keywords'].apply(convert)

In [27]:
movies.head(1)

,movie_id,title,genres,overview,keywords,cast,crew
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","In the 22nd century, a paraplegic Marine is di...","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


#in cast  we are taking only the top 3 actors from each dictionary means we are only taking the top 3 dictionaries and from that also we are taking the only cast name 



In [28]:
def convert3(obj):
    L=[]
    count=0
    for i in ast.literal_eval(obj):
        if count!=3:
            L.append(i['name'])
            count=count+1
        else:
            break   
        
    return L

In [29]:
movies['cast']=movies['cast'].apply(convert3)

In [30]:
import ast

def convert4(obj):
    if obj is None:
        return []

    if isinstance(obj, float) and pd.isna(obj):
        return []

    if isinstance(obj, str):
        try:
            obj = ast.literal_eval(obj)
        except (ValueError, TypeError, SyntaxError):
            return []

    if not isinstance(obj, list):
        return []

    for i in obj:
        if isinstance(i, dict) and i.get("job") == "Director":
            return [i.get("name")]

    return []



In [31]:
movies["crew"] = movies["crew"].apply(convert4)

In [32]:
movies.head(1)

,movie_id,title,genres,overview,keywords,cast,crew
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","In the 22nd century, a paraplegic Marine is di...","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [33]:
# converting the overview also in the list 
movies['overview'].apply(lambda x:x.split())

0       [In, the, 22nd, century,, a, paraplegic, Marin...
1       [Captain, Barbossa,, long, believed, to, be, d...
2       [A, cryptic, message, from, Bond’s, past, send...
3       [Following, the, death, of, District, Attorney...
4       [John, Carter, is, a, war-weary,, former, mili...
                              ...                        
4804    [El, Mariachi, just, wants, to, play, his, gui...
4805    [A, newlywed, couple's, honeymoon, is, upended...
4806    ["Signed,, Sealed,, Delivered", introduces, a,...
4807    [When, ambitious, New, York, attorney, Sam, is...
4808    [Ever, since, the, second, grade, when, he, fi...
Name: overview, Length: 4806, dtype: object

In [34]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [35]:
movies.head(1)

,movie_id,title,genres,overview,keywords,cast,crew
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[In, the, 22nd, century,, a, paraplegic, Marin...","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [36]:
movies['genres']=movies['genres'].apply(lambda x:[i.replace(" ","") for i in x] )
movies['cast']=movies['cast'].apply(lambda x:[i.replace(" ","") for i in x] )
movies['crew']=movies['crew'].apply(lambda x:[i.replace(" ","") for i in x] )
movies['keywords']=movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x] )

In [37]:
movies.head()

,movie_id,title,genres,overview,keywords,cast,crew
0,19995,Avatar,"[Action, Adventure, Fantasy, ScienceFiction]","[In, the, 22nd, century,, a, paraplegic, Marin...","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Captain, Barbossa,, long, believed, to, be, d...","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[Action, Adventure, Crime]","[A, cryptic, message, from, Bond’s, past, send...","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Following, the, death, of, District, Attorney...","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[Action, Adventure, ScienceFiction]","[John, Carter, is, a, war-weary,, former, mili...","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [38]:
movies['tags']=movies['genres']+movies['crew']+movies['cast']+movies['overview']+movies['keywords']

In [39]:
movies['tags'][0]

['Action',
 'Adventure',
 'Fantasy',
 'ScienceFiction',
 'JamesCameron',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d']

In [40]:
new_df=movies[['movie_id' , 'title' , 'tags']]

In [41]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[Action, Adventure, Fantasy, ScienceFiction, J..."
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action, GoreVerbinski, Jo..."
2,206647,Spectre,"[Action, Adventure, Crime, SamMendes, DanielCr..."
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller, ChristopherNo..."
4,49529,John Carter,"[Action, Adventure, ScienceFiction, AndrewStan..."


In [42]:
new_df['tags'][0]

['Action',
 'Adventure',
 'Fantasy',
 'ScienceFiction',
 'JamesCameron',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d']

In [43]:
# converting the tag list into the string 
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14020\3318681842.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))


In [44]:
# converting all tags to the lower alphabet
new_df['tags']=new_df['tags'].apply(lambda x: x.lower())

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14020\2406738864.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags']=new_df['tags'].apply(lambda x: x.lower())


In [45]:
new_df['tags'][0]

'action adventure fantasy sciencefiction jamescameron samworthington zoesaldana sigourneyweaver in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d'

In [59]:
# combining all the tags and taking the most common 5000 wordsa and then creating the vector for each of the tag in the tags columns 
# using the scikit class
# Install scikit-learn if it's missing in the environment
%pip install scikit-learn

from sklearn.feature_extraction.text import CountVectorizer

# Use the vectorizer to build the word-frequency matrix
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [61]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [62]:
# all 5000 words 
# all 5000 words
print(cv.get_feature_names_out())

['000' '007' '10' ... 'zone' 'zoo' 'zooeydeschanel']


In [63]:
# removing the similar words 
# by using stemming 
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import nltk

In [52]:
def stem(text):
    y=[]

    for i in text:
        y.append(ps.stem(i))


    return " ".join(y)    

In [54]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14020\2275729520.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [58]:
new_df['tags'][0]

'action adventur fantasi sciencefict jamescameron samworthington zoesaldana sigourneyweav in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d'

In [64]:
# now we will calculate the distance between the vectors 
# angle distance not the ELUCIDAN distance (distance between the tips of the vectors)

#if distance is more then the similarity will be less and vice versa 

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [67]:
similarity[1]

array([0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
       0.02615329], shape=(4806,))

In [ ]:
# we will recommend like this 
#first if user gives a movie then we fetch the index of the movie from that index we go on to the 
# similarity and then sort the similarity so we got the highest similar top 5and least similar at the end

In [78]:
def recommend(movie):
    movie_index = new_df[new_df['title'].str.lower() == 'movie'].index[0]
    distances=similarity[movie_index]
    movie_list=sorted(enumerate(distances) ,reverse=True,key=lambda x:x[1])[1:6]
    for i in movie_list:
        print(i[0])

In [81]:
def recommend(movie):
    matches = new_df["title"].str.casefold().eq(movie.casefold()).to_numpy().nonzero()[0]

    if len(matches) == 0:
        print(f"Movie not found: {movie}")
        return

    movie_pos = matches[0]
    distances = similarity[movie_pos]

    recommendations = sorted(
        (
            (i, score)
            for i, score in enumerate(distances)
            if i != movie_pos
        ),
        key=lambda item: item[1],
        reverse=True,
    )[:5]

    for i, score in recommendations:
        print(f"{new_df.iloc[i]['title']} — similarity: {score:.3f}")

In [82]:
recommend('Avatar')

Aliens vs Predator: Requiem — similarity: 0.287
Aliens — similarity: 0.269
Falcon Rising — similarity: 0.261
Independence Day — similarity: 0.256
Titan A.E. — similarity: 0.250
